# CS3807 – Deep Learning Laboratory
## Experiment 6 — End-to-End Study of RNN, LSTM and GRU for Sequence Learning and Video Understanding

Complete implementation notebook based on the Experiment 6 handout:
UCI HAR raw inertial signals, RNN/LSTM/GRU comparison, BPTT,
sequence-length analysis, CNN + recurrent video understanding, and
synthetic sequence-to-sequence reversal.


## 1. Imports and Configuration

In [ ]:
# Colab setup and reproducibility

!pip -q install opencv-python

import os
import time
import random
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

SEQUENCE_LENGTH = 128
NUM_FEATURES = 9
NUM_CLASSES = 6
RECURRENT_UNITS = 32
DROPOUT_RATE = 0.2
LEARNING_RATE = 1e-3
BATCH_SIZE = 32
EPOCHS = 30

# Explicit Colab paths
DATA_DIR = Path("/content/data")
HAR_DIR = DATA_DIR / "UCI HAR Dataset"
PLOT_DIR = Path("/content/plots")

DATA_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("Working directory:", Path.cwd())
print("HAR directory:", HAR_DIR)


## 2. Download UCI HAR Dataset

In [ ]:
# Download UCI HAR Dataset — Colab-safe

HAR_URL = "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip"
ZIP_PATH = DATA_DIR / "har.zip"

required_file = HAR_DIR / "train" / "Inertial Signals" / "body_acc_x_train.txt"

if not required_file.exists():
    print("UCI HAR raw files not found.")
    print("Downloading dataset...")

    if not ZIP_PATH.exists():
        urllib.request.urlretrieve(HAR_URL, ZIP_PATH)
        print("Download complete.")
    else:
        print("ZIP already exists:", ZIP_PATH)

    print("Extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(DATA_DIR)

# Recover automatically if the extracted folder is nested differently.
if not required_file.exists():
    candidates = list(DATA_DIR.rglob("body_acc_x_train.txt"))

    if candidates:
        actual_signal_file = candidates[0]
        HAR_DIR = actual_signal_file.parents[2]
        print("Detected dataset at:", HAR_DIR)
    else:
        raise FileNotFoundError(
            "UCI HAR dataset was downloaded/extracted, but "
            "body_acc_x_train.txt could not be found."
        )

print("\nUCI HAR dataset ready.")
print("HAR_DIR:", HAR_DIR)
print("Required file:", HAR_DIR / "train" / "Inertial Signals" / "body_acc_x_train.txt")


In [ ]:
# Verify the exact files required by Section 3

required_files = [
    HAR_DIR / "train" / "Inertial Signals" / "body_acc_x_train.txt",
    HAR_DIR / "train" / "Inertial Signals" / "body_gyro_x_train.txt",
    HAR_DIR / "train" / "Inertial Signals" / "total_acc_x_train.txt",
    HAR_DIR / "train" / "y_train.txt",
    HAR_DIR / "test" / "Inertial Signals" / "body_acc_x_test.txt",
    HAR_DIR / "test" / "y_test.txt",
]

for p in required_files:
    print(f"{'OK' if p.exists() else 'MISSING'}: {p}")

if not all(p.exists() for p in required_files):
    raise FileNotFoundError("One or more required UCI HAR files are missing.")

print("\nDataset structure verified successfully.")


## 3. Load Raw Inertial Signals

In [ ]:
SIGNAL_FILES = [
    "body_acc_x", "body_acc_y", "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z"
]

ACTIVITY_NAMES = [
    "WALKING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS",
    "SITTING", "STANDING", "LAYING"
]

def load_signal_file(split, signal_name):
    path = HAR_DIR / split / "Inertial Signals" / f"{signal_name}_{split}.txt"
    return np.loadtxt(path)

def load_raw_split(split):
    channels = [load_signal_file(split, name) for name in SIGNAL_FILES]
    X = np.stack(channels, axis=-1).astype(np.float32)
    y = np.loadtxt(HAR_DIR / split / f"y_{split}.txt", dtype=int) - 1
    return X, y.astype(np.int64)

X_train_raw, y_train_raw = load_raw_split("train")
X_test_raw, y_test = load_raw_split("test")

print("Train:", X_train_raw.shape, y_train_raw.shape)
print("Test :", X_test_raw.shape, y_test.shape)


## 4. Select Laboratory Subset

In [ ]:
USE_SUBSET = True
SUBSET_SIZE = 3000

if USE_SUBSET:
    rng = np.random.default_rng(SEED)
    selected = []
    base = SUBSET_SIZE // NUM_CLASSES
    remainder = SUBSET_SIZE % NUM_CLASSES

    for cls in range(NUM_CLASSES):
        idx = np.where(y_train_raw == cls)[0]
        n = min(base + (1 if cls < remainder else 0), len(idx))
        selected.extend(rng.choice(idx, size=n, replace=False))

    selected = np.array(selected)
    rng.shuffle(selected)
    X_train_full = X_train_raw[selected]
    y_train_full = y_train_raw[selected]
else:
    X_train_full = X_train_raw
    y_train_full = y_train_raw

print("Selected training:", X_train_full.shape)


## 5. Train / Validation / Test Split and Normalization

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.17647,
    random_state=SEED,
    stratify=y_train_full
)

# Statistics are calculated only from training data.
train_mean = X_train.mean(axis=(0, 1), keepdims=True)
train_std = X_train.std(axis=(0, 1), keepdims=True)
train_std = np.where(train_std < 1e-8, 1.0, train_std)

X_train = (X_train - train_mean) / train_std
X_val = (X_val - train_mean) / train_std
X_test = (X_test_raw - train_mean) / train_std

print("Training :", X_train.shape)
print("Validation:", X_val.shape)
print("Testing  :", X_test.shape)
print("Classes:", NUM_CLASSES)
print("Features per time step:", NUM_FEATURES)
print("Sequence length:", SEQUENCE_LENGTH)


## 6. Class Distribution

In [ ]:
counts = np.bincount(y_train, minlength=NUM_CLASSES)

plt.figure(figsize=(9, 5))
plt.bar(ACTIVITY_NAMES, counts)
plt.xticks(rotation=25, ha="right")
plt.ylabel("Number of sequences")
plt.title("Training Class Distribution")
plt.tight_layout()
plt.savefig(PLOT_DIR / "class_distribution.png", dpi=300)
plt.show()


## 7. Temporal Data Visualization — Plot 1

In [ ]:
def plot_temporal_sequences(X_original, y, classes=(0, 1, 3), channels=(0, 1, 3)):
    fig, axes = plt.subplots(len(classes), 1, figsize=(11, 9), sharex=True)
    axes = np.atleast_1d(axes)

    for ax, cls in zip(axes, classes):
        idx = np.where(y == cls)[0][0]
        for ch in channels:
            ax.plot(
                np.arange(1, SEQUENCE_LENGTH + 1),
                X_original[idx, :, ch],
                label=SIGNAL_FILES[ch]
            )
        ax.set_ylabel(ACTIVITY_NAMES[cls])
        ax.legend(fontsize=8)
        ax.grid(alpha=0.25)

    axes[-1].set_xlabel("Time step")
    fig.suptitle("Temporal Sensor Signals for Representative Activities")
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "plot1_temporal_sensor_signals.png", dpi=300)
    plt.show()

plot_temporal_sequences(X_train_full, y_train_full)


## 8. Numerical RNN Exercise

In [ ]:
x_values = [0.5, 0.7, 0.2]
h = 0.0
Wx, Wh, b = 0.5, 0.8, 0.1
hidden_values = []

for x in x_values:
    h = np.tanh(Wx * x + Wh * h + b)
    hidden_values.append(h)

for i, value in enumerate(hidden_values, 1):
    print(f"h{i} = {value:.6f}")


## 9. BPTT Demonstration

In [ ]:
# Conceptual recurrent graph. During model.fit(), TensorFlow computes
# gradients through the unrolled recurrent computation automatically.

bptt_demo = keras.Sequential([
    layers.Input(shape=(5, 1)),
    layers.SimpleRNN(4),
    layers.Dense(1)
])
bptt_demo.summary()


## 10. RNN / LSTM / GRU Model Builder

In [ ]:
def build_recurrent_model(model_type, sequence_length=SEQUENCE_LENGTH):
    inputs = keras.Input(
        shape=(sequence_length, NUM_FEATURES),
        name="sensor_sequence"
    )

    if model_type == "RNN":
        x = layers.SimpleRNN(RECURRENT_UNITS)(inputs)
    elif model_type == "LSTM":
        x = layers.LSTM(RECURRENT_UNITS)(inputs)
    elif model_type == "GRU":
        x = layers.GRU(RECURRENT_UNITS)(inputs)
    else:
        raise ValueError("model_type must be RNN, LSTM or GRU")

    x = layers.Dropout(DROPOUT_RATE)(x)
    x = layers.Dense(16, activation="relu")(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name=model_type)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

def train_recurrent_model(model_type, Xtr=X_train, ytr=y_train,
                          Xv=X_val, yv=y_val, sequence_length=SEQUENCE_LENGTH):
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)

    model = build_recurrent_model(model_type, sequence_length)
    start = time.perf_counter()

    history = model.fit(
        Xtr, ytr,
        validation_data=(Xv, yv),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=1
    )

    training_time = time.perf_counter() - start
    return model, history, training_time


## 11. Train Vanilla RNN

In [ ]:
rnn_model, rnn_history, rnn_time = train_recurrent_model("RNN")
print(f"RNN training time: {rnn_time:.2f} seconds")


## 12. Train LSTM

In [ ]:
lstm_model, lstm_history, lstm_time = train_recurrent_model("LSTM")
print(f"LSTM training time: {lstm_time:.2f} seconds")


## 13. Train GRU

In [ ]:
gru_model, gru_history, gru_time = train_recurrent_model("GRU")
print(f"GRU training time: {gru_time:.2f} seconds")


## 14. Training and Validation Curves — Plots 2 and 3

In [ ]:
def plot_history(history, model_name):
    epochs = range(1, len(history.history["loss"]) + 1)

    plt.figure(figsize=(9, 5))
    plt.plot(epochs, history.history["loss"], label="Training Loss")
    plt.plot(epochs, history.history["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{model_name}: Training and Validation Loss")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / f"{model_name.lower()}_loss.png", dpi=300)
    plt.show()

    plt.figure(figsize=(9, 5))
    plt.plot(epochs, np.array(history.history["accuracy"]) * 100, label="Training Accuracy")
    plt.plot(epochs, np.array(history.history["val_accuracy"]) * 100, label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.title(f"{model_name}: Training and Validation Accuracy")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / f"{model_name.lower()}_accuracy.png", dpi=300)
    plt.show()

plot_history(rnn_history, "RNN")
plot_history(lstm_history, "LSTM")
plot_history(gru_history, "GRU")


## 15. Test Evaluation — Accuracy, Macro Precision, Macro Recall, Macro F1, Parameters and Training Time

In [ ]:
def evaluate_model(model, model_name, X_eval=X_test, y_eval=y_test, training_time=None):
    probabilities = model.predict(X_eval, verbose=0)
    predictions = np.argmax(probabilities, axis=1)

    result = {
        "Model": model_name,
        "Accuracy (%)": accuracy_score(y_eval, predictions) * 100,
        "Macro Precision (%)": precision_score(y_eval, predictions, average="macro", zero_division=0) * 100,
        "Macro Recall (%)": recall_score(y_eval, predictions, average="macro", zero_division=0) * 100,
        "Macro F1 (%)": f1_score(y_eval, predictions, average="macro", zero_division=0) * 100,
        "Parameters": model.count_params(),
        "Training Time (s)": training_time
    }

    print(f"\n{model_name} Classification Report")
    print(classification_report(
        y_eval, predictions,
        target_names=ACTIVITY_NAMES,
        digits=4,
        zero_division=0
    ))

    return result, predictions, probabilities

rnn_result, rnn_pred, rnn_prob = evaluate_model(rnn_model, "RNN", training_time=rnn_time)
lstm_result, lstm_pred, lstm_prob = evaluate_model(lstm_model, "LSTM", training_time=lstm_time)
gru_result, gru_pred, gru_prob = evaluate_model(gru_model, "GRU", training_time=gru_time)

results_df = pd.DataFrame([rnn_result, lstm_result, gru_result])
display(results_df)


## 16. Confusion Matrices — Plot 4

In [ ]:
def plot_confusion(y_true, y_pred, model_name):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=ACTIVITY_NAMES,
        yticklabels=ACTIVITY_NAMES
    )
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"{model_name} Confusion Matrix")
    plt.xticks(rotation=35, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / f"{model_name.lower()}_confusion_matrix.png", dpi=300)
    plt.show()

plot_confusion(y_test, rnn_pred, "RNN")
plot_confusion(y_test, lstm_pred, "LSTM")
plot_confusion(y_test, gru_pred, "GRU")


## 17. RNN vs LSTM vs GRU — Plot 5

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(results_df))
width = 0.25

ax.bar(x - width, results_df["Accuracy (%)"], width, label="Accuracy")
ax.bar(x, results_df["Macro F1 (%)"], width, label="Macro F1")

param_norm = results_df["Parameters"] / results_df["Parameters"].max() * 100
ax.bar(x + width, param_norm, width, label="Normalized Parameters")

ax.set_xticks(x)
ax.set_xticklabels(results_df["Model"])
ax.set_ylabel("Percentage / Normalized Value")
ax.set_title("RNN vs LSTM vs GRU Performance Comparison")
ax.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "plot5_model_performance_comparison.png", dpi=300)
plt.show()


## 18. Effect of Sequence Length — T ∈ {32, 64, 128}

In [ ]:
def prepare_sequence_length(X, length):
    return X[:, :length, :]

sequence_results = []

for length in [32, 64, 128]:
    Xtr = prepare_sequence_length(X_train, length)
    Xv = prepare_sequence_length(X_val, length)
    Xt = prepare_sequence_length(X_test, length)

    for model_type in ["RNN", "LSTM", "GRU"]:
        tf.keras.backend.clear_session()
        tf.random.set_seed(SEED)

        model = build_recurrent_model(model_type, sequence_length=length)

        start = time.perf_counter()
        history = model.fit(
            Xtr, y_train,
            validation_data=(Xv, y_val),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            verbose=0
        )
        training_time = time.perf_counter() - start

        pred = np.argmax(model.predict(Xt, verbose=0), axis=1)
        f1 = f1_score(y_test, pred, average="macro", zero_division=0)

        sequence_results.append({
            "Sequence Length": length,
            "Model": model_type,
            "Test F1 (%)": f1 * 100,
            "Training Time (s)": training_time
        })

sequence_df = pd.DataFrame(sequence_results)
display(sequence_df)


### Plot 6 — Sequence Length vs Test F1-score

In [ ]:
plt.figure(figsize=(9, 5))

for model_type in ["RNN", "LSTM", "GRU"]:
    subset = sequence_df[sequence_df["Model"] == model_type]
    plt.plot(
        subset["Sequence Length"],
        subset["Test F1 (%)"],
        marker="o",
        label=model_type
    )

plt.xlabel("Sequence Length")
plt.ylabel("Test Macro F1 (%)")
plt.title("Effect of Sequence Length on Test F1-score")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(PLOT_DIR / "plot6_sequence_length_vs_f1.png", dpi=300)
plt.show()


# Part II — Video Understanding Using CNN + RNN

The handout recommends a small UCF101 subset. Set `UCF101_DIR` to the extracted UCF101 directory before running this section.


In [ ]:
UCF101_DIR = Path("data/UCF-101")
VIDEO_CLASSES = ["Basketball", "Biking", "Walking", "Running", "TennisSwing"]
VIDEO_FRAMES = 10
VIDEO_IMAGE_SIZE = (224, 224)
VIDEO_MAX_PER_CLASS = 20

print("UCF101 directory:", UCF101_DIR)


## 19. Video Frame Sampling

In [ ]:
def sample_video_frames(video_path, num_frames=VIDEO_FRAMES):
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        cap.release()
        return None

    indices = np.linspace(0, total_frames - 1, num_frames).astype(int)
    frames = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()

        if not ok:
            cap.release()
            return None

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, VIDEO_IMAGE_SIZE)
        frames.append(frame)

    cap.release()
    return np.asarray(frames, dtype=np.uint8)


## 20. MobileNetV2 CNN Feature Extraction

In [ ]:
cnn_feature_extractor = tf.keras.applications.MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3),
    pooling="avg"
)
cnn_feature_extractor.trainable = False
CNN_FEATURE_DIM = cnn_feature_extractor.output_shape[-1]

print("CNN feature dimension:", CNN_FEATURE_DIM)


In [ ]:
def collect_video_paths(root_dir, classes, max_per_class):
    paths, labels = [], []

    for label, class_name in enumerate(classes):
        class_dir = root_dir / class_name
        if not class_dir.exists():
            print(f"Missing class directory: {class_dir}")
            continue

        video_files = sorted([
            p for p in class_dir.iterdir()
            if p.suffix.lower() in {".avi", ".mp4", ".mov", ".mkv"}
        ])

        for path in video_files[:max_per_class]:
            paths.append(path)
            labels.append(label)

    return paths, np.asarray(labels, dtype=np.int64)

def extract_video_feature_sequence(video_path):
    frames = sample_video_frames(video_path)
    if frames is None:
        return None

    frames = tf.keras.applications.mobilenet_v2.preprocess_input(
        frames.astype(np.float32)
    )
    return cnn_feature_extractor(frames, training=False).numpy()

video_X = np.empty((0, VIDEO_FRAMES, CNN_FEATURE_DIM), dtype=np.float32)
video_y = np.empty((0,), dtype=np.int64)
valid_video_paths = []

if UCF101_DIR.exists():
    video_paths, video_labels = collect_video_paths(
        UCF101_DIR, VIDEO_CLASSES, VIDEO_MAX_PER_CLASS
    )

    features, labels, valid_paths = [], [], []

    for path, label in zip(video_paths, video_labels):
        try:
            f = extract_video_feature_sequence(path)
            if f is not None:
                features.append(f)
                labels.append(label)
                valid_paths.append(path)
        except Exception as exc:
            print("Skipping", path, "because:", exc)

    if features:
        video_X = np.stack(features).astype(np.float32)
        video_y = np.asarray(labels, dtype=np.int64)
        valid_video_paths = valid_paths

print("Video feature tensor:", video_X.shape)
print("Video labels:", video_y.shape)


## 21. Video Sample Frames — Plot 7

In [ ]:
if valid_video_paths:
    sample_frames = sample_video_frames(valid_video_paths[0])

    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    for ax, frame, i in zip(axes.ravel(), sample_frames, range(len(sample_frames))):
        ax.imshow(frame)
        ax.set_title(f"Frame {i + 1}")
        ax.axis("off")

    fig.suptitle(f"Sampled Frames — {valid_video_paths[0].parent.name}")
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "plot7_video_sample_frames.png", dpi=300)
    plt.show()
else:
    print("UCF101 features are not available. Set UCF101_DIR and rerun.")


## 22. Video Train / Validation / Test Split

In [ ]:
video_train_X = video_val_X = video_test_X = None
video_train_y = video_val_y = video_test_y = None

if len(video_X) > 0:
    video_train_X, temp_X, video_train_y, temp_y = train_test_split(
        video_X, video_y,
        test_size=0.30,
        random_state=SEED,
        stratify=video_y
    )

    video_val_X, video_test_X, video_val_y, video_test_y = train_test_split(
        temp_X, temp_y,
        test_size=0.50,
        random_state=SEED,
        stratify=temp_y
    )

    print("Training:", video_train_X.shape)
    print("Validation:", video_val_X.shape)
    print("Testing:", video_test_X.shape)


## 23. CNN–LSTM / CNN–GRU Model

In [ ]:
def build_video_model(model_type="LSTM"):
    inputs = keras.Input(shape=(VIDEO_FRAMES, CNN_FEATURE_DIM))

    if model_type == "LSTM":
        x = layers.LSTM(32)(inputs)
    elif model_type == "GRU":
        x = layers.GRU(32)(inputs)
    else:
        raise ValueError("model_type must be LSTM or GRU")

    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(len(VIDEO_CLASSES), activation="softmax")(x)

    model = keras.Model(inputs, outputs, name=f"CNN_{model_type}")
    model.compile(
        optimizer=keras.optimizers.Adam(LEARNING_RATE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


In [ ]:
video_model = None
video_history = None
video_training_time = None
video_pred = None

if video_train_X is not None:
    video_model = build_video_model("LSTM")

    start = time.perf_counter()
    video_history = video_model.fit(
        video_train_X, video_train_y,
        validation_data=(video_val_X, video_val_y),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=1
    )
    video_training_time = time.perf_counter() - start

    print(f"Video model training time: {video_training_time:.2f} seconds")
else:
    print("Video dataset not available.")


## 24. Video Training and Validation Curves — Plot 8

In [ ]:
if video_history is not None:
    epochs = range(1, len(video_history.history["loss"]) + 1)

    plt.figure(figsize=(9, 5))
    plt.plot(epochs, video_history.history["loss"], label="Training Loss")
    plt.plot(epochs, video_history.history["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("CNN–LSTM Video Training and Validation Loss")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "plot8_video_loss.png", dpi=300)
    plt.show()

    plt.figure(figsize=(9, 5))
    plt.plot(epochs, np.array(video_history.history["accuracy"]) * 100, label="Training Accuracy")
    plt.plot(epochs, np.array(video_history.history["val_accuracy"]) * 100, label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.title("CNN–LSTM Video Training and Validation Accuracy")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "plot8_video_accuracy.png", dpi=300)
    plt.show()


## 25. Video Confusion Matrix — Plot 9

In [ ]:
if video_model is not None:
    video_prob = video_model.predict(video_test_X, verbose=0)
    video_pred = np.argmax(video_prob, axis=1)

    print(classification_report(
        video_test_y, video_pred,
        target_names=VIDEO_CLASSES,
        digits=4,
        zero_division=0
    ))

    cm = confusion_matrix(video_test_y, video_pred)

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=VIDEO_CLASSES,
        yticklabels=VIDEO_CLASSES
    )
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("CNN–LSTM Video Confusion Matrix")
    plt.xticks(rotation=35, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "plot9_video_confusion_matrix.png", dpi=300)
    plt.show()

    print("Video Accuracy:",
          accuracy_score(video_test_y, video_pred) * 100)
    print("Video Macro F1:",
          f1_score(video_test_y, video_pred, average="macro", zero_division=0) * 100)


# Part III — Sequence-to-Sequence Learning

Synthetic reversal task:
`[1, 4, 7, 2] → [2, 7, 4, 1]`


## 26. Generate Synthetic Reversal Dataset

In [ ]:
SEQ_VOCAB_SIZE = 20
SEQ_LENGTH = 4
SEQ_DATASET_SIZE = 5000
START_TOKEN = 0

rng = np.random.default_rng(SEED)

seq_X = rng.integers(
    1, SEQ_VOCAB_SIZE,
    size=(SEQ_DATASET_SIZE, SEQ_LENGTH)
)
seq_y = np.flip(seq_X, axis=1)

decoder_input = np.zeros_like(seq_y)
decoder_input[:, 0] = START_TOKEN
decoder_input[:, 1:] = seq_y[:, :-1]

(
    encoder_train, encoder_test,
    decoder_train, decoder_test,
    target_train, target_test
) = train_test_split(
    seq_X, decoder_input, seq_y,
    test_size=0.20,
    random_state=SEED
)

(
    encoder_train, encoder_val,
    decoder_train, decoder_val,
    target_train, target_val
) = train_test_split(
    encoder_train, decoder_train, target_train,
    test_size=0.20,
    random_state=SEED
)

print("Encoder train:", encoder_train.shape)
print("Decoder train:", decoder_train.shape)
print("Target train :", target_train.shape)

for i in range(5):
    print("Input:", seq_X[i], "Target:", seq_y[i])


## 27. Encoder–Decoder LSTM

In [ ]:
EMBED_DIM = 32
LATENT_DIM = 64

encoder_inputs = keras.Input(
    shape=(SEQ_LENGTH,), dtype="int32", name="encoder_input"
)
encoder_embedding = layers.Embedding(
    SEQ_VOCAB_SIZE, EMBED_DIM
)(encoder_inputs)

encoder_lstm = layers.LSTM(LATENT_DIM, return_state=True)
_, state_h, state_c = encoder_lstm(encoder_embedding)

decoder_inputs = keras.Input(
    shape=(SEQ_LENGTH,), dtype="int32", name="decoder_input"
)
decoder_embedding = layers.Embedding(
    SEQ_VOCAB_SIZE, EMBED_DIM
)(decoder_inputs)

decoder_lstm = layers.LSTM(
    LATENT_DIM, return_sequences=True, return_state=True
)
decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=[state_h, state_c]
)

decoder_outputs = layers.Dense(
    SEQ_VOCAB_SIZE, activation="softmax"
)(decoder_outputs)

seq2seq_model = keras.Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

seq2seq_model.compile(
    optimizer=keras.optimizers.Adam(LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

seq2seq_model.summary()


## 28. Train Seq2Seq Model

In [ ]:
seq2seq_history = seq2seq_model.fit(
    [encoder_train, decoder_train],
    target_train[..., np.newaxis],
    validation_data=(
        [encoder_val, decoder_val],
        target_val[..., np.newaxis]
    ),
    epochs=30,
    batch_size=64,
    verbose=1
)


## 29. Seq2Seq Training and Validation Loss

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(seq2seq_history.history["loss"], label="Training Loss")
plt.plot(seq2seq_history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Seq2Seq Training and Validation Loss")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(PLOT_DIR / "seq2seq_training_validation_loss.png", dpi=300)
plt.show()


## 30. Seq2Seq Evaluation — Token and Sequence Accuracy

In [ ]:
# Teacher-forced evaluation gives per-token predictions.
test_probs = seq2seq_model.predict(
    [encoder_test, decoder_test],
    verbose=0
)
test_pred = np.argmax(test_probs, axis=-1)

token_accuracy = np.mean(test_pred == target_test)
sequence_accuracy = np.mean(
    np.all(test_pred == target_test, axis=1)
)

print(f"Token Accuracy    : {token_accuracy * 100:.2f}%")
print(f"Sequence Accuracy : {sequence_accuracy * 100:.2f}%")
print(f"Training Loss     : {seq2seq_history.history['loss'][-1]:.6f}")
print(f"Validation Loss   : {seq2seq_history.history['val_loss'][-1]:.6f}")

print("\nFive test examples:")
for i in range(min(5, len(encoder_test))):
    print(f"{i+1}.")
    print("Input    :", encoder_test[i])
    print("Target   :", target_test[i])
    print("Predicted:", test_pred[i])


## 31. Consolidated Results

In [ ]:
print("RNN / LSTM / GRU Results")
display(results_df)

print("\nSequence Length Results")
display(sequence_df)

if video_model is not None:
    video_result = pd.DataFrame([{
        "Model": "CNN-LSTM",
        "Accuracy (%)": accuracy_score(video_test_y, video_pred) * 100,
        "Macro Precision (%)": precision_score(
            video_test_y, video_pred, average="macro", zero_division=0
        ) * 100,
        "Macro Recall (%)": recall_score(
            video_test_y, video_pred, average="macro", zero_division=0
        ) * 100,
        "Macro F1 (%)": f1_score(
            video_test_y, video_pred, average="macro", zero_division=0
        ) * 100,
        "Parameters": video_model.count_params(),
        "Training Time (s)": video_training_time
    }])
    print("\nCNN-LSTM Video Results")
    display(video_result)

print("\nSeq2Seq Results")
print(f"Token Accuracy: {token_accuracy * 100:.2f}%")
print(f"Sequence Accuracy: {sequence_accuracy * 100:.2f}%")


## 32. Final Submission Checklist

- Raw UCI HAR inertial signals used
- Input tensor `(N, 128, 9)`
- Six activity classes
- Train/validation/test protocol
- Training-only normalization
- Temporal visualization
- Numerical RNN exercise
- BPTT demonstration
- Vanilla RNN
- LSTM
- GRU
- Training/validation loss and accuracy curves
- Accuracy, macro precision, macro recall and macro F1
- Confusion matrices
- Parameter count and training time
- RNN/LSTM/GRU comparison
- Sequence lengths 32, 64 and 128
- CNN feature extraction
- Ten-frame video representation
- CNN–LSTM video pipeline
- Video training/validation curves
- Video confusion matrix
- Synthetic reversal Seq2Seq task
- Five Seq2Seq examples
- Token accuracy
- Sequence accuracy
- Consolidated results

All numerical performance values are produced by execution and should not be manually assumed.


## Colab execution note

Run the notebook from the top after connecting to a Colab runtime. The setup cell installs OpenCV, the download cell downloads/extracts UCI HAR, and the verification cell checks the exact raw signal files before Section 3 loads them.
